In [ ]:
import os

import numpy as np
import pandas as pd
from scipy.stats import binom, false_discovery_control

os.makedirs("../../result/HEK293T_HCT116/ranking/", exist_ok=True)

### Gene symbol mapping

In [ ]:
hgnc = pd.read_csv(
    "../../data/HGNC/hgnc_complete_set.txt",
    sep="\t",
    low_memory=False,
)
hgnc = hgnc[hgnc["status"] == "Approved"]
approved_symbols = set(hgnc["symbol"])

with_entrez = hgnc.dropna(subset=["entrez_id"])
entrez2symbol = dict(
    zip(with_entrez["entrez_id"].astype(int), with_entrez["symbol"])
)

old2symbol = {}
for column in ["prev_symbol", "alias_symbol"]:
    for symbol, blob in zip(hgnc["symbol"], hgnc[column]):
        if not isinstance(blob, str):
            continue
        for old in blob.split("|"):
            old = old.strip()
            if old == "" or old in approved_symbols:
                continue
            old2symbol.setdefault(old, set()).add(symbol)
old2symbol = {
    old: next(iter(v)) for old, v in old2symbol.items() if len(v) == 1
}

### Load the bioplex as bait-prey pairs

In [ ]:
def load_bioplex(
    infile,
):
    df = pd.read_csv(infile, sep="\t")
    out = pd.DataFrame({
        "bait": df["GeneA"].map(entrez2symbol),
        "prey": df["GeneB"].map(entrez2symbol),
    })
    out["bait"] = (
        out["bait"]
        .fillna(df["SymbolA"].map(old2symbol))
        .fillna(df["SymbolA"])
    )
    out["prey"] = (
        out["prey"]
        .fillna(df["SymbolB"].map(old2symbol))
        .fillna(df["SymbolB"])
    )
    out = out[out["bait"] != out["prey"]]
    return out.drop_duplicates()


hek = load_bioplex(
    "../../data/BioPlex/BioPlex_293T_Network_10K_Dec_2019.tsv"
)
hct = load_bioplex(
    "../../data/BioPlex/BioPlex_HCT116_Network_5.5K_Dec_2019.tsv"
)

### Restrict to baits present in both cell lines and count prey captures

In [ ]:
shared_baits = set(hek["bait"]) & set(hct["bait"])
hek_shared = hek[hek["bait"].isin(shared_baits)]
hct_shared = hct[hct["bait"].isin(shared_baits)]
e_hek = len(hek_shared)
e_hct = len(hct_shared)

# Per-prey counts of capturing shared baits
counts = pd.DataFrame({
    "n_HEK293T": hek_shared["prey"].value_counts(),
    "n_HCT116": hct_shared["prey"].value_counts(),
}).fillna(0).astype(int)
counts.index.name = "gene"

### Degree-imbalance test against the network-wide proportion

In [ ]:
# Global success probability under the null
p_hek = e_hek / (e_hek + e_hct)

# Two-sided binomial on the HEK293T share of captures
k = counts["n_HEK293T"].to_numpy()
n = k + counts["n_HCT116"].to_numpy()
lower = binom.cdf(k, n, p_hek)
upper = binom.sf(k - 1, n, p_hek)
counts["p_degree"] = np.minimum(1.0, 2.0 * np.minimum(lower, upper))
counts["log2FC"] = np.log2(
    ((counts["n_HEK293T"] + 0.5) / e_hek)
    / ((counts["n_HCT116"] + 0.5) / e_hct)
)
counts["fdr"] = false_discovery_control(
    counts["p_degree"].to_numpy(), method="bh"
)
# make -0.0 as 0.0
counts["score"] = 0.0 - np.log10(counts["p_degree"])

### Rank and write

In [ ]:
ranking = (
    counts.reset_index()
    .sort_values(
        ["score", "log2FC", "gene"],
        ascending=[False, False, True],
        key=lambda s: s.abs() if s.name == "log2FC" else s,
    )
    .reset_index(drop=True)
)
ranking["rank"] = np.arange(1, len(ranking) + 1)
ranking = ranking[[
    "gene",
    "score",
    "rank",
    "log2FC",
    "n_HEK293T",
    "n_HCT116",
    "p_degree",
    "fdr",
]]
ranking.to_csv(
    "../../result/HEK293T_HCT116/ranking/AP-MS_degree.ranking",
    sep="\t",
    index=False,
)